In [1]:
import os
import shutil
import numpy as np

import keras

from delft.sequenceLabelling import Sequence
from delft.sequenceLabelling.config import ModelConfig
# from delft.sequenceLabelling.reader import load_data_and_labels_crf_file
from delft.sequenceLabelling.models import get_model
from delft.sequenceLabelling.preprocess import Preprocessor
from delft.utilities.Embeddings import Embeddings
from delft.sequenceLabelling.wrapper import CONFIG_FILE_NAME, PROCESSOR_FILE_NAME

2025-09-09 13:23:57.754039: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-09 13:23:57.755618: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-09 13:23:57.767619: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-09 13:23:57.921681: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757381038.079727  646720 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757381038.08

In [2]:
model_name = "grobid-date-BidLSTM_CRF"
dir_path = "data/models/sequenceLabelling/"
model_path = os.path.join(dir_path, model_name)
out_dir = "./export/"

In [3]:
# Load model and get inner/base models
seq = Sequence(model_name)

# get the model config and overwrite the model config in seq
dir_path = 'data/models/sequenceLabelling/'
model_path = os.path.join(dir_path, model_name)
model_config = ModelConfig.load(os.path.join(model_path, CONFIG_FILE_NAME))
seq.model_config = model_config

seq.embeddings = Embeddings(
    model_config.embeddings_name, resource_registry=seq.registry, use_ELMo=model_config.use_ELMo)

seq.p = Preprocessor.load(os.path.join(dir_path, model_config.model_name, PROCESSOR_FILE_NAME))

seq.model = get_model(
    model_config, seq.p, ntags=len(seq.p.vocab_tag), load_pretrained_weights=False, local_path=model_path)

seq.model.summary(line_length=100, expand_nested=True, show_trainable=True)


BidLSTM_CRF


2025-09-09 13:24:05.687332: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "crf_wrapper"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Layer (type)                          ┃ Output Shape                  ┃        Param # ┃ Traina… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ functional (Functional)               │ (None, None, 100)             │        392,850 │    Y    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ char_input (InputLayer)          │ (None, None, 30)              │              0 │    -    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ time_distributed                 │ (None, None, 30, 25)          │          1,750 │    Y    │
│ (TimeDistributed)                     │                               │                │         │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ word_input (InputLayer)          │ (None, None, 300)             │              0 │    -    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ time_distributed_1               │ (None, None, 50)              │         10,200 │    Y    │
│ (TimeDistributed)                     │                               │                │         │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ concatenate (Concatenate)        │ (None, None, 350)             │              0 │    -    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ dropout (Dropout)                │ (None, None, 350)             │              0 │    -    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ bidirectional_1 (Bidirectional)  │ (None, None, 200)             │        360,800 │    Y    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ dropout_1 (Dropout)              │ (None, None, 200)             │              0 │    -    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ dense (Dense)                    │ (None, None, 100)             │         20,100 │    Y    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ length_input (InputLayer)        │ (None, 1)                     │              0 │    -    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│    └ length_passthrough (TakeFirst)   │ (None, None, 100)             │              0 │    -    │
├───────────────────────────────────────┼───────────────────────────────┼────────────────┼─────────┤
│ crf (CRF)                             │ ((None, None), (None, None,   │            770 │    Y    │
│                                       │ 7), (None), (7, 7))           │                │         │
└───────────────────────────────────────┴───────────────────────────────┴────────────────┴─────────┘

 Total params: 393,620 (1.50 MB)

 Trainable params: 393,620 (1.50 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
for t in ([(v.path, v.shape.as_list()) for v in seq.model.weights]):
    print(t)
    

('time_distributed/char_embeddings/embeddings', [70, 25])
('time_distributed_1/bidirectional/forward_lstm/lstm_cell/kernel', [25, 100])
('time_distributed_1/bidirectional/forward_lstm/lstm_cell/recurrent_kernel', [25, 100])
('time_distributed_1/bidirectional/forward_lstm/lstm_cell/bias', [100])
('time_distributed_1/bidirectional/backward_lstm/lstm_cell/kernel', [25, 100])
('time_distributed_1/bidirectional/backward_lstm/lstm_cell/recurrent_kernel', [25, 100])
('time_distributed_1/bidirectional/backward_lstm/lstm_cell/bias', [100])
('bidirectional_1/forward_lstm_1/lstm_cell/kernel', [350, 400])
('bidirectional_1/forward_lstm_1/lstm_cell/recurrent_kernel', [100, 400])
('bidirectional_1/forward_lstm_1/lstm_cell/bias', [400])
('bidirectional_1/backward_lstm_1/lstm_cell/kernel', [350, 400])
('bidirectional_1/backward_lstm_1/lstm_cell/recurrent_kernel', [100, 400])
('bidirectional_1/backward_lstm_1/lstm_cell/bias', [400])
('dense/kernel', [200, 100])
('dense/bias', [100])
('crf_wrapper/crf/t

In [ ]:
# dir(seq.model.weights[0])

In [5]:
import h5py
filepath = os.path.join(model_path, 'model_weights.hdf5')
with h5py.File(filepath, 'r') as f:
    f.visititems(lambda name, obj: print(name, list(obj.shape)) if isinstance(obj, h5py.Dataset) else None)

crf/chain_kernel:0 [7, 7]
crf/crf/dense_1/bias:0 [7]
crf/crf/dense_1/kernel:0 [100, 7]
crf/left_boundary:0 [7]
crf/right_boundary:0 [7]
model/bidirectional_1/backward_lstm_1/lstm_cell_5/bias:0 [400]
model/bidirectional_1/backward_lstm_1/lstm_cell_5/kernel:0 [350, 400]
model/bidirectional_1/backward_lstm_1/lstm_cell_5/recurrent_kernel:0 [100, 400]
model/bidirectional_1/forward_lstm_1/lstm_cell_4/bias:0 [400]
model/bidirectional_1/forward_lstm_1/lstm_cell_4/kernel:0 [350, 400]
model/bidirectional_1/forward_lstm_1/lstm_cell_4/recurrent_kernel:0 [100, 400]
model/dense/bias:0 [100]
model/dense/kernel:0 [200, 100]
model/time_distributed/embeddings:0 [70, 25]
model/time_distributed_1/backward_lstm/lstm_cell_2/bias:0 [100]
model/time_distributed_1/backward_lstm/lstm_cell_2/kernel:0 [25, 100]
model/time_distributed_1/backward_lstm/lstm_cell_2/recurrent_kernel:0 [25, 100]
model/time_distributed_1/forward_lstm/lstm_cell_1/bias:0 [100]
model/time_distributed_1/forward_lstm/lstm_cell_1/kernel:0 [25